<img src="../images/bwHPC_Logo_cmyk.svg" width="200" /> <img src="../images/HochschuleEsslingen_Logo_RGB_DE.png" width="200" /> <img src="../images/Konstanz_Logo.svg" width="200" /> <img src="../images/KIT_Logo.png" width="200" />

# Exceptions, Context Managers and Modules

Handling errors with `try`/`except`, managing resources with the `with` statement, and organising code into modules that can be imported.

---

## Contents

1. [Exceptions](#exceptions)
2. [With, \_\_enter\_\_, \_\_exit\_\_ and contextmanager](#with-enter-exit-and-contextmanager)
3. [Module and import](#module-and-import)

<a id="exceptions"></a>
## 1. Exceptions

Python provides the keywords `try`, `except` and `raise` to support error handling.

With `raise` one may throw an error. This error may be fetched with a surrounding try-block. If the error is not caught, it will be written to the console and the program terminated.

If an instruction within the try-block throws directly or indirectly (e.g. via a called function) another error, this may be fetched using one or multiple except-blocks after the try-block. Each except-instruction is checked and the first exception instruction fitting the raised error (see `isinstance` in [Comparison of data types](02_Operators_and_Control_Structures.ipynb#comparison-of-data-types)) is executed. In case an instruction within the exception instruction throws yet another error, another try-block is required surrounding this try-block.

After the except instruction a try-block may have two additional instructions: `else` and `finally`. The else block is only executed if the try-block finished without error. The finally block will always be executed, independent of whether an exception was thrown, or whether it was caught.

One may derive new Error/Exceptions based on existing Error classes. See [Built-In Exceptions](https://pythonbasics.org/try-except/)

In [ ]:
raise TypeError() # creates an error object of type TypeError and throws

In [ ]:
raise TypeError # short form for raise TypeError()

In [ ]:
raise TypeError("Parameter x hase wrong value y") # throws the TypeError with additional Argument

In [ ]:
a = [0, 1]
b = 5

try:
    if a < b:
        print("unreachable due to Exception")
except Exception as e: # catches all objects of type Exception and Objects, whose type is derived from Exception
    print(str(type(e)) + ": " + str(e))

In [ ]:
def throw_exception(i):
    if type(i) != int:
        raise TypeError("Parameter is not of type int")
    if i < 0 or i > 10:
        raise ValueError("Parameter is not within the range 0-10")
    if i < 6:
        raise Exception("Error happened")

try:
    #throw_exception("test")
    #throw_exception(11)
    #throw_exception(5)
    throw_exception(6)
except TypeError as te:
    print("TypeError was raised: " + str(te))
except ValueError as ve:
    print("ValueError was raised: " + str(ve))
except: # catches all errors, but does not provide the error as instance
    print("Unknown error")
else:
    print("No error happened")
finally:
    print("Will always be executed")
    # break, continue or return within the finally-Block prevents raising of unknown errors
    # If both try- and the finally block contain a return, only the one from finally will be executed

In [ ]:
class TestException(Exception):
    pass

raise TestException("Just a test")

In [ ]:
class TestException(Exception):
    pass

for e in (TypeError("Parameter is not of type int"), ValueError("Parameter is not in range 0-10"), TestException("Just a Test")):
    try:
        raise e
    except (TypeError, ValueError) as e: # There may be more than one error class caught by this except instruction
        print("Type- oder ValueError")
    except BaseException as e: # all Exceptions derive from BaseException
        print(e)
        raise e # rethrow Exception

In [ ]:
class TestException(Exception):
    pass

try:
    print("test")
    raise TypeError
except TypeError as e:
    raise ValueError() # in except and in finally new errors are linked to the existing error
else:
    raise ValueError()
finally:
    raise ValueError() # in except and in finally new errors are linked to the existing error

In [ ]:
class TestException(Exception):
    pass

try:
    print("test")
    raise TypeError
except TypeError as e:
    raise ValueError() from None # from None: prohibits linking this error to existing error
else:
    raise ValueError()
finally:
    raise ValueError() from None # from None: prohibits linking this error to existing error

<a id="with-enter-exit-and-contextmanager"></a>
## 2. With, \_\_enter\_\_, \_\_exit\_\_ and contextmanager

Context managers allow you to allocate and release resources using the `with`-statement.
Both an object as well as a function may be a context manager. The object / the function has to contain instructions to create and destroy of a context.
A context e.g. may be a file or a database connection.
Upon creating the context the file, or the database connection is opened.
Upon closing the context the file, or the database connection are terminated, as well.
Any exceptions are handled for you by the context manager, freeing you of the pain of writing boilerplate code.

In [ ]:
try:
    with open("../files/test.csv") as file:
        print("file closed: "+ str(file.closed))
        #raise EOFError("Aborted reading from file")
        print(file.readline())
finally:
    print("file closed: "+ str(file.closed))

In [ ]:
class Writer:
    def __init__(self, file_name):
        self.file_name = file_name
    
    def __enter__(self): # Will be called by the with-statement
        print("Opening file")
        self.file = open(self.file_name, 'r')
        return self.file # return value from __enter__ will be assigned using the as keyword
    
    def __exit__(self, exception_type, exception_value, traceback): # will be called upon exiting the with block
        if exception_type is not None:
            print("There was an error, we do not handle it")
            self.file.close()
            return False # but rather pass it on
        print("Closing the file")
        self.file.close()
        return True

try:
    with Writer("../files/test.csv") as file:
        print("file closed: "+ str(file.closed))
        #raise EOFError("Aborted reading from file")
        print(file.readline())
finally:
    print("file closed: "+ str(file.closed))

In [ ]:
from contextlib import contextmanager

class Writer:
    def __init__(self, file):
        self.file_name = file
    
    @contextmanager
    def open_and_close(self):
        try:
            print("Opening file")
            self.file = open(self.file_name, 'r')
            yield self.file # thanks to @contextmanager yield operates just like __exit__
        finally:
            print("Closing file")
            self.file.close()

try:
    with Writer("../files/test.csv").open_and_close() as file:
        print("file closed: "+ str(file.closed))
        #raise EOFError("Aborted reading")
        print(file.readline())
finally:
    print("file closed: "+ str(file.closed))

<a id="module-and-import"></a>
## 3. Module and import

Modules allows encapsulating functions (and data) in separate files (with the usual file extension ".py"). Using the keyword `import` these functions may be included and used in other Python sources. The strength of Python lies in the amount and versatility of available modules.

In [ ]:
import platform

x = platform.system()
print(x)

In [ ]:
import platform

print(dir(platform)) # dir lists all attributes (functions/methods and variables) in an object

In [ ]:
import platform

for e in dir(platform):
    x = getattr(platform, e)
    if (callable(x)): # if x callable (a function or a class containing the method __call__
        print(x)

In [ ]:
import platform as p # Rename the module as p for less typing

x = p.system()
print(x)

In [ ]:
from platform import system # Just imports a single function/variable from a larger module

print(system())

In [ ]:
from platform import system as s # both may be combined for just the needed functionality and less typing

print(s())

Your own modules may be used just the any existing python module:

In [ ]:
import os

### Let's create a module for You and write it to disk
module_as_string = """# import test module
def test():
    print("This is a test")
"""

with open("test.py", "w") as file:
    file.write(module_as_string)

### and use this very module
import test

test.test()

os.remove("test.py")